In [1]:
from google.colab import files
import pandas as pd
uploaded = files.upload()

Saving 1k_stories_100_genre.csv to 1k_stories_100_genre.csv


In [2]:
df = pd.read_csv("1k_stories_100_genre.csv")
df.head()

,id,title,story,genre
0,457580,The Chronicles of the Cosmic Rift,"In the year 2250, Earth had made significant s...",Science Fiction
1,297904,Eldoria's Enchanted Whispers,"In a land far away, where the sun shone bright...",Fantasy
2,620436,Echoes of Whispered Shadows,"Once upon a time, in a small, tranquil town ca...",Mystery
3,634687,Emerald Amulet Chronicles Revealed,"Once upon a time in the 16th century, a small ...",Historical Adventure
4,513427,The Shadows of St. Augustine,In the sun-drenched coastal city of St. August...,Thriller


We had problems running the full experiment on Google Colab because the notebook was freezing and taking too long to finish. Because of that, we had to optimize the experiment while keeping the same idea. We still used the top 15 genres and 5 prompt variations, but reduced the number of generated samples to 1 per prompt. This gave us 75 generations instead of 375, which made the notebook much faster and more stable on Colab.

In [3]:
GENRE_COL = "genre"
TEXT_COL = "story"
TITLE_COL = "title"

In [4]:
genre_counts = df[GENRE_COL].value_counts()

In [5]:
TOP_GENRES = genre_counts.head(15).index.tolist()

In [6]:
print("Top 15 genres:")
for i, g in enumerate(TOP_GENRES, 1):
    print(f"{i:2}. {g} ({genre_counts[g]} stories)")

Top 15 genres:
 1. Historical Adventure (20 stories)
 2. Fantasy (10 stories)
 3. Science Fiction (10 stories)
 4. Mystery (10 stories)
 5. Thriller (10 stories)
 6. Historical Fiction (10 stories)
 7. Adventure (10 stories)
 8. Horror (10 stories)
 9. Comedy (10 stories)
10. Crime (10 stories)
11. Dystopian (10 stories)
12. Cyberpunk (10 stories)
13. Steampunk (10 stories)
14. Post-Apocalyptic (10 stories)
15. Fairy Tale (10 stories)


In [7]:
N_SHOTS = 1
MAX_EXAMPLE_CHARS = 180

In [8]:
few_shot_pool = {}

for genre in TOP_GENRES:
    subset = (
        df[df[GENRE_COL] == genre]
        .dropna(subset=[TEXT_COL])
        .sample(frac=1, random_state=42)
        .reset_index(drop=True)
    )

    few_shot_pool[genre] = []
    for _, row in subset.head(N_SHOTS).iterrows():
        few_shot_pool[genre].append({
            "title": str(row.get(TITLE_COL, "")),
            "text": str(row[TEXT_COL])[:MAX_EXAMPLE_CHARS].replace("\n", " ")
        })

In [9]:
few_shot_pool[TOP_GENRES[0]]

[{'title': 'Emerald Amulet Chronicles Revealed',
  'text': 'Once upon a time in the 16th century, a small village nestled in the heart of the English countryside, far from the maddening crowd. The villagers, led by the wise and benevolent M'}]

### Prompt Variations

All five prompts describe the same scenario, an autumn morning written in a melancholic style, but differ in how the instruction is phrased. P1 is the most direct and minimal version, providing only the essential request. P2 conveys the same meaning through paraphrased wording, allowing us to examine whether small lexical changes affect the generated output. P3 introduces a specific literary tone by emphasizing sadness and nostalgia. P4 focuses more strongly on the emotional aspect and encourages a poetic response. Finally, P5 provides the most detailed guidance by explicitly requesting literary devices such as metaphors and sensory imagery, which is expected to produce richer and more descriptive text.

In [10]:
BASE_PROMPTS = [
    "Write a description of an autumn morning in a melancholic style.",
    "Create a short text portraying an autumn morning with a melancholic atmosphere.",
    "Describe an autumn morning using a sad nostalgic literary tone.",
    "Write a poetic paragraph about an autumn morning filled with melancholy.",
    "Write a literary description of an autumn morning using metaphors and sensory imagery.",
]

In [11]:
def build_few_shot_prompt(genre: str, base_prompt: str, n_shots: int = 2) -> str:
    examples = few_shot_pool[genre][:n_shots]

    lines = [
        f"You are writing in the genre: {genre}.",
        "Use the following real examples from the dataset only as style and genre references.",
        "Do not copy them. Write a new original text for the task.",
        "",
        "### Dataset examples"
    ]

    for i, ex in enumerate(examples, 1):
        lines.append(f"Example {i} - {genre}")
        if ex["title"]:
            lines.append(f"Title: {ex['title']}")
        lines.append(f"Text: {ex['text']}")
        lines.append("")

    lines.extend([
        "### Task",
        base_prompt,
        "Keep the same genre influence as the examples, but the topic must stay an autumn morning.",
        "Generated text:"
    ])

    return "\n".join(lines)

In [12]:
all_prompts = []
for genre in TOP_GENRES:
    for prompt_idx, base_prompt in enumerate(BASE_PROMPTS, start=1):
        all_prompts.append({
            "target_genre": genre,
            "prompt_idx": prompt_idx,
            "base_prompt": base_prompt,
            "prompt": build_few_shot_prompt(genre, base_prompt, n_shots=N_SHOTS)
        })

In [13]:
print(all_prompts[0]["prompt"])

You are writing in the genre: Historical Adventure.
Use the following real examples from the dataset only as style and genre references.
Do not copy them. Write a new original text for the task.

### Dataset examples
Example 1 - Historical Adventure
Title: Emerald Amulet Chronicles Revealed
Text: Once upon a time in the 16th century, a small village nestled in the heart of the English countryside, far from the maddening crowd. The villagers, led by the wise and benevolent M

### Task
Write a description of an autumn morning in a melancholic style.
Keep the same genre influence as the examples, but the topic must stay an autumn morning.
Generated text:


In [14]:
from transformers import pipeline
import torch, gc

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: Tesla T4


In [15]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [16]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

tokenizer = AutoTokenizer.from_pretrained("google/t5gemma-2b-2b-prefixlm-it")
model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/t5gemma-2b-2b-prefixlm-it",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)


config.json:   0%|          | 0.00/3.26k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/577 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/68.1k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/732 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

In [17]:
def generate_texts(prompt_entries, samples_per_prompt=1):
    rows = []

    for idx, entry in enumerate(prompt_entries, start=1):
        print(f"Generating {idx}/{len(prompt_entries)} | {entry['target_genre']} | prompt {entry['prompt_idx']}")

        for sample_id in range(1, samples_per_prompt + 1):
            messages = [{"role": "user", "content": entry["prompt"]}]

            inputs = tokenizer.apply_chat_template(
                messages,
                return_tensors="pt",
                return_dict=True,
                add_generation_prompt=True,
            ).to(model.device)

            with torch.no_grad():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=80,
                    min_new_tokens=20,
                    temperature=0.7,
                    top_p=0.9,
                    do_sample=True,
                )

            generated_only = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

            if not generated_only:
                with torch.no_grad():
                    output_ids = model.generate(
                        **inputs,
                        max_new_tokens=60,
                        do_sample=False,
                    )
                generated_only = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

            rows.append({
                "target_genre": entry["target_genre"],
                "prompt_idx": entry["prompt_idx"],
                "sample_id": sample_id,
                "base_prompt": entry["base_prompt"],
                "generated": generated_only,
            })

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return rows

In [18]:
SAMPLES_PER_PROMPT = 1

In [19]:
results_df = pd.DataFrame(
    generate_texts(all_prompts, SAMPLES_PER_PROMPT)
)

results_df["generated"] = results_df["generated"].fillna("").astype(str).str.strip()
empty_count = (results_df["generated"] == "").sum()

if empty_count > 0:
    print(f"Warning: {empty_count} empty generations were removed before evaluation.")
    results_df = results_df[results_df["generated"] != ""].reset_index(drop=True)

results_df.to_csv("t5gemma_generations.csv", index=False)
print("Generated rows:", len(results_df))
results_df[["target_genre", "prompt_idx", "sample_id", "generated"]].head()

Generating 1/75 | Historical Adventure | prompt 1
Generating 2/75 | Historical Adventure | prompt 2
Generating 3/75 | Historical Adventure | prompt 3
Generating 4/75 | Historical Adventure | prompt 4
Generating 5/75 | Historical Adventure | prompt 5
Generating 6/75 | Fantasy | prompt 1
Generating 7/75 | Fantasy | prompt 2
Generating 8/75 | Fantasy | prompt 3
Generating 9/75 | Fantasy | prompt 4
Generating 10/75 | Fantasy | prompt 5
Generating 11/75 | Science Fiction | prompt 1
Generating 12/75 | Science Fiction | prompt 2
Generating 13/75 | Science Fiction | prompt 3
Generating 14/75 | Science Fiction | prompt 4
Generating 15/75 | Science Fiction | prompt 5
Generating 16/75 | Mystery | prompt 1
Generating 17/75 | Mystery | prompt 2
Generating 18/75 | Mystery | prompt 3
Generating 19/75 | Mystery | prompt 4
Generating 20/75 | Mystery | prompt 5
Generating 21/75 | Thriller | prompt 1
Generating 22/75 | Thriller | prompt 2
Generating 23/75 | Thriller | prompt 3
Generating 24/75 | Thriller

,target_genre,prompt_idx,sample_id,generated
0,Historical Adventure,1,1,"The first light of dawn, a pale ghost of gold,..."
1,Historical Adventure,2,1,"The first rays of dawn, pale and hesitant, kis..."
2,Historical Adventure,3,1,"The first light of dawn, a timid fingertip of ..."
3,Historical Adventure,4,1,"The sun, a bruised ember in the pale October s..."
4,Historical Adventure,5,1,"The sun, a bruised ruby in the eastern sky, bl..."


After each text is generated, a zero-shot Natural Language Inference (NLI) classifier is used to evaluate whether the output matches the intended genre. The classifier is based on the `facebook/bart-large-mnli` model and does not require additional training on the dataset. Instead, each genre is represented through a descriptive label (e.g., *"fantasy with magic, enchanted forests and mythical creatures"* or *"science fiction with space travel and futuristic technology"*). The generated text is compared against all candidate genre descriptions, and the model predicts the most likely genre by measuring semantic similarity between the text and the genre labels. This approach allows automatic verification of genre consistency and provides an objective way to assess whether the generated story aligns with the genre specified in the prompt.

In [20]:
GENRE_HINT_MAP = {
    "Historical Adventure": "historical adventure set in past centuries with quests and kingdoms",
    "Science Fiction": "science fiction with space travel, futuristic technology and alien worlds",
    "Fantasy": "fantasy with magic, enchanted forests and mythical creatures",
    "Mystery": "mystery with detectives, secrets and unsolved crimes",
    "Thriller": "thriller with danger, betrayal and high-stakes tension",
    "Historical Fiction": "historical fiction set in a specific era with period-accurate detail",
    "Adventure": "adventure with exploration, brave heroes and unknown lands",
    "Horror": "horror with ghosts, haunted places and terrifying events",
    "Comedy": "comedy with humorous characters and funny situations",
    "Crime": "crime story with criminals, investigations and urban danger",
    "Dystopian": "dystopian society with oppressive regimes and ruined civilizations",
    "Cyberpunk": "cyberpunk with neon cities, hacking and human-machine fusion",
    "Steampunk": "steampunk with Victorian-era technology and mechanical inventions",
    "Post-Apocalyptic": "post-apocalyptic world after disaster, survival and collapsed society",
    "Fairy Tale": "fairy tale with magical kingdoms, princes, witches and moral lessons",
}

In [21]:
GENRE_HINTS = {g: GENRE_HINT_MAP.get(g, g.lower() + " story") for g in TOP_GENRES}

In [22]:
candidate_labels = list(GENRE_HINTS.values())

In [23]:
label_to_genre = {v: k for k, v in GENRE_HINTS.items()}

In [24]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if torch.cuda.is_available() else -1,
)

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [25]:
def classify_genre(text: str) -> str:
    text = str(text).strip()
    if not text:
        return "EMPTY_GENERATION"

    result = classifier(
        text[:400],
        candidate_labels=candidate_labels,
        hypothesis_template="This text is a {}.",
        multi_label=False
    )
    return label_to_genre[result["labels"][0]]

In [26]:
results_df["predicted_genre"] = results_df["generated"].apply(classify_genre)
results_df["genre_match"] = results_df["target_genre"] == results_df["predicted_genre"]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [28]:
results_df[["target_genre", "predicted_genre", "genre_match", "generated"]].head()

,target_genre,predicted_genre,genre_match,generated
0,Historical Adventure,Historical Fiction,False,"The first light of dawn, a pale ghost of gold,..."
1,Historical Adventure,Historical Fiction,False,"The first rays of dawn, pale and hesitant, kis..."
2,Historical Adventure,Historical Fiction,False,"The first light of dawn, a timid fingertip of ..."
3,Historical Adventure,Historical Fiction,False,"The sun, a bruised ember in the pale October s..."
4,Historical Adventure,Historical Fiction,False,"The sun, a bruised ruby in the eastern sky, bl..."


In [29]:
overall = results_df["genre_match"].mean()

In [30]:
print(f"Genre Match Accuracy: {overall:.2%}")

Genre Match Accuracy: 18.67%


In [31]:
import numpy as np
import torch
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline, GPT2LMHeadModel, GPT2Tokenizer

In [33]:
texts = results_df["generated"].dropna().astype(str).str.strip()
texts = texts[texts != ""].tolist()

**Cosine similarity** was used to measure the semantic similarity between the generated texts. First, each text was converted into a vector representation using the `all-MiniLM-L6-v2` embedding model. Cosine similarity was then computed for every pair of texts, and the average score was calculated. Higher similarity values indicate that the outputs convey similar meanings and themes despite differences in prompt wording.

In [34]:
embed_device = "cuda" if torch.cuda.is_available() else "cpu"
embed_model = SentenceTransformer("all-MiniLM-L6-v2", device=embed_device)

embeddings = embed_model.encode(texts, batch_size=32, show_progress_bar=True)
sim_matrix = cosine_similarity(embeddings)

upper = sim_matrix[np.triu_indices(len(sim_matrix), k=1)]
avg_cosine = float(np.mean(upper))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

**Sentiment analysis** was applied to all generated texts using a pretrained sentiment classification model. For each text, the model predicted whether the sentiment was positive or negative and assigned a confidence score. To create a single numerical scale, positive predictions were recorded as positive values, while negative predictions were assigned negative values. The average sentiment score was then calculated to measure the overall emotional tone of the generated texts, while the standard deviation was used to assess the variation in sentiment across different outputs. Since all prompts requested a melancholic description of an autumn morning, the generated texts were expected to exhibit predominantly negative sentiment.

In [35]:
sent_pipe = pipeline(
    "sentiment-analysis",
    device=0 if torch.cuda.is_available() else -1,
)

sent_scores = []

for t in texts:
    r = sent_pipe(t[:512])[0]
    score = r["score"] if r["label"] == "POSITIVE" else -r["score"]
    sent_scores.append(score)

sentiment_avg = float(np.mean(sent_scores))
sentiment_var = float(np.std(sent_scores))

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

**Perplexity** was computed using a pretrained GPT-2 language model as a measure of text fluency and linguistic coherence. For each generated text, the model estimated the likelihood of the observed word sequence and converted the prediction loss into a perplexity score. Lower perplexity values correspond to more natural and predictable language, whereas higher values indicate reduced fluency or increased linguistic complexity. The variability of perplexity scores across generated texts was assessed using the standard deviation, providing insight into the robustness of the model's language generation under different prompt formulations.

In [36]:
gpt2_tok = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_mdl = GPT2LMHeadModel.from_pretrained("gpt2")

if torch.cuda.is_available():
    gpt2_mdl = gpt2_mdl.to("cuda")

gpt2_mdl.eval()

def perplexity(text):
    enc = gpt2_tok(text, return_tensors="pt", truncation=True, max_length=512)

    if torch.cuda.is_available():
        enc = {k: v.to("cuda") for k, v in enc.items()}

    with torch.no_grad():
        out = gpt2_mdl(**enc, labels=enc["input_ids"])

    return torch.exp(out.loss).item()

ppl_scores = [perplexity(t) for t in texts]
ppl_var = float(np.std(ppl_scores))

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


The generated texts were evaluated using a combination of semantic, stylistic, and linguistic metrics. Cosine similarity was used to measure semantic consistency, sentiment metrics assessed the preservation of the intended melancholic tone, and perplexity variation evaluated the stability of language fluency across outputs. **Style consistency** measured how frequently the same genre was predicted among the generated texts, while genre match accuracy quantified the alignment between the intended and predicted genres.

In [37]:
style_counts = Counter(results_df["predicted_genre"].tolist())
style_consistency = style_counts.most_common(1)[0][1] / len(texts)

In [38]:
from IPython.display import display

metrics = {
    "cosine_similarity": avg_cosine,
    "sentiment_avg": sentiment_avg,
    "sentiment_variation": sentiment_var,
    "perplexity_variation": ppl_var,
    "style_consistency": style_consistency,
    "genre_match_accuracy": overall,
}

df_metrics = pd.DataFrame(list(metrics.items()), columns=["Metric", "Value"])

display(
    df_metrics
    .style
    .hide(axis="index")
    .format({"Value": "{:.4f}"})
)

Metric,Value
cosine_similarity,0.6112
sentiment_avg,0.2359
sentiment_variation,0.8782
perplexity_variation,11.5722
style_consistency,0.6800
genre_match_accuracy,0.1867


T5Gemma achieved relatively high text similarity (0.6112) and good style consistency (0.6800), indicating stable generation behavior. However, the genre match accuracy was low (0.1867), suggesting that the model struggled to follow the target genre instructions. Overall, the model produced consistent outputs but showed weaker genre adherence.

In [39]:
files.download("t5gemma_generations.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>